# Phase 2: Feature Engineering

This notebook performs feature engineering for match predictor.

In [ ]:
import pandas as pd
import numpy as np

# Load all data
results_df = pd.read_csv('../data/results.csv')
elo_df = pd.read_csv('../data/elo_ratings.csv')
former_names_df = pd.read_csv('../data/former_names.csv')

## 1. Standardize Team Names

Historical teams like 'Czech Republic' to 'Czechia', 'Soviet Union' to 'Russia', etc.

In [ ]:
# Create name mapping
name_mapping = dict(zip(former_names_df['former'], former_names_df['current']))

def standardize_team(team_name):
    return name_mapping.get(team_name, team_name)

# Apply to results_df and elo_df
results_df['home_team'] = results_df['home_team'].apply(standardize_team)
results_df['away_team'] = results_df['away_team'].apply(standardize_team)
elo_df['team'] = elo_df['team'].apply(standardize_team)

results_df.head()

## 2. Calculate Target Variable ('result')

- 0: Home Win
- 1: Draw
- 2: Away Win

In [ ]:
def get_result(row):
    if row['home_score'] > row['away_score']:
        return 0
    elif row['home_score'] < row['away_score']:
        return 2
    else:
        return 1

results_df['result'] = results_df.apply(get_result, axis=1)
results_df['result'].value_counts()

## 3. Compute Rolling 5-Match Form Features

In [ ]:
team_history = {}

home_form = []
away_form = []

home_gf = []
away_gf = []

home_gc = []
away_gc = []

home_gd = []
away_gd = []

# Sort data by date to ensure correct order
results_df = results_df.sort_values('date').reset_index(drop=True)

for _, row in results_df.iterrows():
    home = row['home_team']
    away = row['away_team']

    if home not in team_history:
        team_history[home] = []
    if away not in team_history:
        team_history[away] = []

    # Get last 5 matches
    home_last5 = team_history[home][-5:]
    away_last5 = team_history[away][-5:]

    # Calculate features
    home_form.append(sum(m['p'] for m in home_last5))
    away_form.append(sum(m['p'] for m in away_last5))

    hgf = sum(m['gf'] for m in home_last5)
    agf = sum(m['gf'] for m in away_last5)
    home_gf.append(hgf)
    away_gf.append(agf)

    hgc = sum(m['gc'] for m in home_last5)
    agc = sum(m['gc'] for m in away_last5)
    home_gc.append(hgc)
    away_gc.append(agc)

    home_gd.append(hgf - hgc)
    away_gd.append(agf - agc)

    # Update history after prediction
    hp = 3 if row['home_score'] > row['away_score'] else (1 if row['home_score'] == row['away_score'] else 0)
    ap = 3 if row['away_score'] > row['home_score'] else (1 if row['home_score'] == row['away_score'] else 0)
    
    if not pd.isna(row['home_score']): # Only add actual played matches
        team_history[home].append({'p': hp, 'gf': row['home_score'], 'gc': row['away_score']})
        team_history[away].append({'p': ap, 'gf': row['away_score'], 'gc': row['home_score']})

# Assign to DataFrame
results_df['home_form_points_last5'] = home_form
results_df['away_form_points_last5'] = away_form

results_df['home_goals_scored_last5'] = home_gf
results_df['away_goals_scored_last5'] = away_gf

results_df['home_goals_conceded_last5'] = home_gc
results_df['away_goals_conceded_last5'] = away_gc

results_df['home_goal_difference_last5'] = home_gd
results_df['away_goal_difference_last5'] = away_gd

results_df.head()

## 4. Merge Elo Ratings

In [ ]:
# Add year
results_df['year'] = pd.to_datetime(results_df['date']).dt.year

# Merge home Elo
results_df = results_df.merge(elo_df, left_on=['year', 'home_team'], right_on=['year', 'team'], how='left')
results_df.rename(columns={'rating': 'home_elo'}, inplace=True)
results_df.drop('team', axis=1, inplace=True)

# Merge away Elo
results_df = results_df.merge(elo_df, left_on=['year', 'away_team'], right_on=['year', 'team'], how='left')
results_df.rename(columns={'rating': 'away_elo'}, inplace=True)
results_df.drop('team', axis=1, inplace=True)

# Calculate Elo difference
results_df['elo_difference'] = results_df['home_elo'] - results_df['away_elo']

print(f"Before cleaning missing Elo: {len(results_df)} rows")
results_df = results_df[results_df['year'] >= 1901]
results_df.dropna(subset=['home_elo', 'away_elo'], inplace=True)
print(f"After cleaning: {len(results_df)} rows")
results_df.head()

In [ ]:
# Save processed features
results_df.to_csv('../data/features_with_elo_cleaned.csv', index=False)
print('Saved: features_with_elo_cleaned.csv!')